#  Notebook 2 — YOLOv8 Training

This notebook focuses on training an object detection model using **YOLOv8**, a fast and efficient deep learning framework designed for real-time performance and ease of use.

steps :
- Set up the training configuration
- Load and prepare your dataset
- Train the model 

The workflow is designed to be simple, modular, and beginner-friendly, allowing you to experiment with different settings 



In [2]:
# ============================================================
#  CONFIG
# ============================================================

DATA_YAML   = "data_local.yaml"  #  path to fixed yaml from Notebook 1
MODEL_SIZE  = "yolov8m.pt"       #  yolov8n.pt (fastest) | yolov8s.pt | yolov8m.pt (balanced) | yolov8l.pt | yolov8x.pt (best)
PROJECT_DIR = "."             #  output folder
RUN_NAME    = "runs/exp3"            #  experiment name

# --- Training ---
EPOCHS      = 1       #   
IMG_SIZE    = 640      #  640 (standard) | 416 (faster) | 1280 (slower)
BATCH_SIZE  = 8        #  
WORKERS     = 0        #  0 on Windows | 4+ on Linux
PATIENCE    = 20       #  early stopping patience (epochs without improvement)
LR0         = 0.01     #  initial learning rate
LRF         = 0.001    #  final learning rate (after cosine decay)

# --- Class imbalance ---
USE_CLASS_WEIGHTS = True  #  True = handle car:bike 25:1 imbalance

# --- Resume ---
RESUME = False  #  True = resume last interrupted run

CLASSES = [
    "traffic light", "traffic sign", "car", "person", "bus",
    "truck", "rider", "bike", "motor", "train", "banner", "tuktuk"
]

print("Config set!")
print(f"   Model      : {MODEL_SIZE}")
print(f"   Epochs     : {EPOCHS}")
print(f"   Image size : {IMG_SIZE}px")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Classes    : {len(CLASSES)}")

Config set!
   Model      : yolov8m.pt
   Epochs     : 1
   Image size : 640px
   Batch size : 8
   Classes    : 12


## Cell 1 — Environment Check

In [3]:
import torch
from ultralytics import YOLO
import ultralytics

print(f"PyTorch      : {torch.__version__}")
print(f"Ultralytics  : {ultralytics.__version__}")

print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")

# Time estimate
import os, yaml
with open(DATA_YAML) as f: data = yaml.safe_load(f)
train_imgs = len(os.listdir(data['train'])) if os.path.exists(data['train']) else 0
steps = train_imgs // BATCH_SIZE
mins  = steps * 0.04 / 60  # ~40ms/step on RTX 4060
print(f"\nTime estimate:")
print(f"   Train images : {train_imgs:,}")
print(f"   Steps/epoch  : {steps:,}")
print(f"   ~{mins:.0f} min/epoch × {EPOCHS} epochs = ~{mins*EPOCHS/60:.1f} hrs")

PyTorch      : 2.5.1+cu121
Ultralytics  : 8.4.30
CUDA         : True
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM         : 6.9 GB free / 8.0 GB total

Time estimate:
   Train images : 48,329
   Steps/epoch  : 6,041
   ~4 min/epoch × 1 epochs = ~0.1 hrs


## Cell 2 — Compute Class Weights (for imbalance)

In [4]:
# import os
# import numpy as np
# from collections import Counter
# import yaml

# with open(DATA_YAML) as f: data = yaml.safe_load(f)
# lbl_dir = os.path.join(os.path.dirname(data['train']), "..", "labels").replace("images", "labels")
# lbl_dir = data['train'].replace("images", "labels")

# counts = Counter()
# if os.path.exists(lbl_dir):
#     for lbl_file in os.listdir(lbl_dir):
#         if not lbl_file.endswith('.txt'): continue
#         with open(os.path.join(lbl_dir, lbl_file)) as f:
#             for line in f:
#                 parts = line.strip().split()
#                 if parts: counts[int(parts[0])] += 1

# total = sum(counts.values())
# # Inverse frequency weights — rare classes get higher weight
# weights = []
# print(f"{'Class':<15} {'Count':>8} {'Weight':>8}")
# print("-" * 35)
# for i, cls in enumerate(CLASSES):
#     count = counts.get(i, 1)
#     w = total / (len(CLASSES) * max(count, 1))
#     weights.append(round(w, 4))
#     flag = "" if w > 2 else ""
#     print(f"{cls:<15} {count:>8,} {w:>8.3f}{flag}")

# print(f"\nClass weights: {weights}")
# print("(higher = model pays more attention to that class)")

##  Cell 3 — Train YOLOv8

This cell runs the full training pipeline using your configured settings. It loads a pretrained YOLOv8 model, fine-tunes it on your dataset, and saves checkpoints along with training metrics.

Key highlights:
- Automatically downloads pretrained weights (first run only)
- Supports GPU acceleration (if available)
- Includes built-in augmentations to improve generalization
- Handles class imbalance through loss weighting and data augmentation
- Saves both best and last model checkpoints


In [5]:
from ultralytics import YOLO

# Load pretrained YOLOv8 
model = YOLO(MODEL_SIZE)
print(f" Loaded: {MODEL_SIZE}")
print(f" Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

# Train
results = model.train(
    data      = DATA_YAML,        # dataset config
    epochs    = EPOCHS,           # total epochs
    imgsz     = IMG_SIZE,         # image size
    batch     = BATCH_SIZE,       # batch size
    workers   = WORKERS,          # dataloader workers
    patience  = PATIENCE,         # early stopping
    lr0       = LR0,              # initial LR
    lrf       = LRF,              # final LR
    project   = PROJECT_DIR,      # output folder
    name      = RUN_NAME,         # experiment name
    resume    = RESUME,           # resume from last checkpoint
    device    = 0 if torch.cuda.is_available() else 'cpu',
    # Class imbalance handling
    cls       = 0.5,              # classification loss weight
    # Augmentations (help with class imbalance)
    mosaic    = 1.0,              # mosaic augmentation (0=off, 1=full)
    mixup     = 0.1,              # mixup augmentation
    copy_paste= 0.1,              # copy-paste augmentation (helps rare classes)
    # Logging
    plots     = True,             # save training plots
    save      = True,             # save checkpoints
    save_period = 10,              # save checkpoint every N epochs
    verbose   = True,
)

print("\n  Training complete!")
print(f"   Best model : {PROJECT_DIR}/{RUN_NAME}/weights/best.pt")
print(f"   Last model : {PROJECT_DIR}/{RUN_NAME}/weights/last.pt")

 Loaded: yolov8m.pt
 Parameters: 25,902,640
New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.30  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937,

## Cell 3B — Focal Loss Trainer (Ablation Study)

Replaces YOLOv8's default BCE classification loss with **Focal Loss** (α=0.25, γ=2.0).

**Why:** Dataset has severe class imbalance (car:bike ≈ 22:1). BCE is dominated by easy car examples. Focal Loss down-weights easy negatives and forces the model to focus on hard minority examples (bike, tuktuk, train).

**Ablation:** Run Cell 3 (baseline BCE) → compare per-class AP50 → run Cell 3B (Focal) → compare. The delta on minority classes is your publishable contribution.

| Hyperparameter | Value | Note |
|---|---|---|
| α (alpha) | 0.25 | Down-weights easy negatives |
| γ (gamma) | 2.0 | Standard from Lin et al. 2017 |
| All other params | Same as Cell 3 | Controlled ablation |


In [7]:
# ============================================================
#  CELL 3B — Focal Loss YOLOv8 Trainer
#  Replaces BCE cls loss with Focal Loss (α=0.25, γ=2.0)
#  Keep all other hyperparams identical to Cell 3 for clean ablation.
# ============================================================

import torch
import torch.nn.functional as F
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss

# ---- 1. Focal Loss module ----
class FocalLoss(torch.nn.Module):
    """
    Focal Loss: FL(pt) = -alpha * (1 - pt)^gamma * log(pt)
    Lin et al., 'Focal Loss for Dense Object Detection', ICCV 2017.
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        loss = self.alpha * (1.0 - pt) ** self.gamma * bce
        return loss   # keep reduction='none' — v8DetectionLoss handles reduction


# ---- 2. Subclass v8DetectionLoss, swap only self.bce ----
class FocalDetectionLoss(v8DetectionLoss):
    def __init__(self, model, tal_topk=10):
        super().__init__(model, tal_topk)
        self.bce = FocalLoss(alpha=0.25, gamma=2.0)
        print("[FocalDetectionLoss] cls loss → Focal Loss (α=0.25, γ=2.0)")


# ---- 3. Custom trainer — patch loss via on_train_start callback ----
#  Avoids overriding _setup_train (signature varies across ultralytics versions)
class FocalLossTrainer(DetectionTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.add_callback('on_train_start', self._inject_focal_loss)

    def _inject_focal_loss(self, trainer):
        trainer.loss_fn = FocalDetectionLoss(trainer.model)
        print("[FocalLossTrainer] Loss patched → FocalDetectionLoss")

    def criterion(self, preds, batch):
        return self.loss_fn(preds, batch)


# ============================================================
#  RUN — Focal Loss experiment
# ============================================================
FOCAL_RUN_NAME = "runs/exp_focal"   # separate dir — keeps baseline intact

model_focal = YOLO(MODEL_SIZE)
print(f" Loaded: {MODEL_SIZE}")
print(f" Parameters: {sum(p.numel() for p in model_focal.model.parameters()):,}")

model_focal.train(
    trainer     = FocalLossTrainer,
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    workers     = WORKERS,
    patience    = PATIENCE,
    lr0         = LR0,
    lrf         = LRF,
    project     = PROJECT_DIR,
    name        = FOCAL_RUN_NAME,
    resume      = False,
    device      = 0 if torch.cuda.is_available() else 'cpu',
    cls         = 0.5,
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.1,
    plots       = True,
    save        = True,
    save_period = 10,
    verbose     = True,
)

print("\n Focal Loss training complete!")
print(f"   Best model : {PROJECT_DIR}/{FOCAL_RUN_NAME}/weights/best.pt")


 Loaded: yolov8m.pt
 Parameters: 25,902,640
New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.30  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937,

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Cell 3C — Ablation: Baseline vs Focal Loss per-class AP50

Run this **after both Cell 3 (baseline) and Cell 3B (focal) complete**.
Produces a side-by-side per-class AP50 table — your paper's Table 1.


In [ ]:
# ============================================================
#  CELL 3C — Ablation Results Table
#  Compares per-class AP50 between BCE baseline and Focal Loss
# ============================================================

import torch
import numpy as np
from ultralytics import YOLO

BASELINE_BEST = f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
FOCAL_BEST    = f"{PROJECT_DIR}/{FOCAL_RUN_NAME}/weights/best.pt"

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def get_per_class_ap(model_path):
    model = YOLO(model_path)
    metrics = model.val(
        data    = DATA_YAML,
        imgsz   = IMG_SIZE,
        batch   = BATCH_SIZE,
        device  = DEVICE,
        verbose = False,
    )
    # ap_class_index and ap give per-class AP50
    ap50_per_class = metrics.box.ap50          # shape: (num_classes,)
    return ap50_per_class

print("Validating baseline model...")
ap_baseline = get_per_class_ap(BASELINE_BEST)

print("Validating focal loss model...")
ap_focal = get_per_class_ap(FOCAL_BEST)

# --- Print ablation table ---
print("\n" + "=" * 62)
print(f"{'Class':<15} {'BCE AP50':>10} {'Focal AP50':>12} {'Delta':>8}")
print("=" * 62)

minority = ['bike', 'tuktuk', 'train', 'rider', 'bus']  # your 22:1 minority classes

for i, cls in enumerate(CLASSES):
    b = ap_baseline[i] if i < len(ap_baseline) else float('nan')
    f = ap_focal[i]    if i < len(ap_focal)    else float('nan')
    delta = f - b
    flag = " ← minority" if cls in minority else ""
    sign = "+" if delta >= 0 else ""
    print(f"{cls:<15} {b:>10.4f} {f:>12.4f} {sign}{delta:>7.4f}{flag}")

print("=" * 62)
print(f"{'mAP50 (all)':<15} {np.mean(ap_baseline):>10.4f} {np.mean(ap_focal):>12.4f} "
      f"{np.mean(ap_focal)-np.mean(ap_baseline):>+8.4f}")

# Minority-only summary
minority_idx = [CLASSES.index(c) for c in minority if c in CLASSES]
m_base  = np.mean([ap_baseline[i] for i in minority_idx])
m_focal = np.mean([ap_focal[i]    for i in minority_idx])
print(f"{'mAP50 (minority)':<15} {m_base:>10.4f} {m_focal:>12.4f} {m_focal-m_base:>+8.4f}")
print("=" * 62)
print("\n Paper claim: Focal Loss (α=0.25, γ=2.0) improves minority-class AP50")
print(f" by {(m_focal-m_base)*100:+.2f}pp vs BCE baseline on Egyptian traffic dataset.")


## Cell 4 — Resume Training (if interrupted)

In [ ]:
# import os
# import torch
# from ultralytics import YOLO

# LAST_CKPT = "runs/egypt_yolo/weights/last.pt"

# if not os.path.exists(LAST_CKPT):
#     print(f"Checkpoint not found: {LAST_CKPT}")
# else:
#     model_r = YOLO(LAST_CKPT)

#     model_r.train(
#         data   = DATA_YAML,
#         epochs = 30,   
#         resume = True,
#         workers=0,
#         device = 0 if torch.cuda.is_available() else 'cpu',
#     )

#     print("Resumed training complete!")

New https://pypi.org/project/ultralytics/8.4.33 available  Update with 'pip install -U ultralytics'
WARNING model 'runs\egypt_yolo\weights\last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.30  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=

RuntimeError: CUDA error: resource already mapped
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Cell 5 — Quick Validation After Training

In [ ]:
import os

BEST_MODEL = f"" 

if not os.path.exists(BEST_MODEL):
    print(f"Model not found: {BEST_MODEL}")
else:
    model_val = YOLO(BEST_MODEL)
    metrics = model_val.val(data=DATA_YAML, imgsz=IMG_SIZE, batch=BATCH_SIZE,
                             device=0 if torch.cuda.is_available() else 'cpu')
    print(f"\n Validation Results:")
    print(f"   mAP50     : {metrics.box.map50:.4f}")
    print(f"   mAP50-95  : {metrics.box.map:.4f}")
    print(f"   Precision : {metrics.box.mp:.4f}")
    print(f"   Recall    : {metrics.box.mr:.4f}")

Ultralytics 8.4.30  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 93 layers, 25,846,708 parameters, 0 gradients, 78.7 GFLOPs



FileNotFoundError: Dataset 'data_local.yaml' images not found, missing path 'C:\Users\hp\Desktop\proj2\dataset\val\val\images'
Note dataset download directory is 'C:\Users\hp\Desktop\proj2\datasets'. You can update this in 'C:\Users\hp\AppData\Roaming\Ultralytics\settings.json'